# TA 실험 1~3: rolling OOF 교정 · 잔차 LSTM · Ridge 제거

- 2022~2024 rolling OOF만으로 교정값과 앙상블 가중치를 결정합니다.
- 2025 라벨은 마지막 평가 전까지 사용하지 않습니다.
- 입력은 LE1B 16채널, 공식 station 좌표/고도, 날짜·태양 계산값으로 제한합니다.

In [ ]:
!pip install -q catboost==1.2.8
import torch, catboost, pandas as pd
print('torch:', torch.__version__, 'catboost:', catboost.__version__)

In [ ]:
# Mountpoint must not already contain files 오류를 피하는 안전한 마운트
import os
from google.colab import drive

DRIVE_MOUNT = '/content/drive'
if not os.path.ismount(DRIVE_MOUNT):
    if os.path.isdir(DRIVE_MOUNT) and os.listdir(DRIVE_MOUNT):
        DRIVE_MOUNT = '/content/drive_ta_exp123'
    drive.mount(DRIVE_MOUNT, force_remount=False)
MYDRIVE = os.path.join(DRIVE_MOUNT, 'MyDrive')
print('MyDrive:', MYDRIVE)

In [ ]:
from pathlib import Path

BRANCH = 'agent/ta-exp123-rolling-residual'
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_DIR = Path('/content/SME_DATA_ta_exp123')
MASTER_CSV = Path(MYDRIVE) / 'SME_DATA/processed_station_features/final_train_dataset_19to25_master.csv'
SHORTTERM_CSV = Path(MYDRIVE) / 'SME_DATA/processed_station_features/shortterm_12to14_data/incremental_12to14_tables/shortterm_long_2019to2025.csv'
OUTPUT_DIR = Path(MYDRIVE) / 'SME_DATA/processed_station_features/model_experiments_19to25/ta_exp123_rolling_residual'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(MASTER_CSV)
print(SHORTTERM_CSV)
print(OUTPUT_DIR)

In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / '.git').exists():
        raise RuntimeError(f'{REPO_DIR}가 존재하지만 Git 저장소가 아닙니다. 다른 REPO_DIR을 지정하세요.')
    current = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if current != BRANCH:
        raise RuntimeError(f'기존 clone의 브랜치가 {current}입니다. REPO_DIR을 바꾸거나 {BRANCH}를 checkout하세요.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

SCRIPT = REPO_DIR / 'scripts/experiment_ta_calibration_residual_ablation.py'
BASELINE = REPO_DIR / 'scripts/train_dualbranch_ta_baseline.py'
for path in [SCRIPT, BASELINE, MASTER_CSV, SHORTTERM_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)
print('실행 파일 및 입력 파일 확인 완료')

In [ ]:
import sys, collections

cmd = [
    sys.executable, '-u', str(SCRIPT),
    '--master-csv', str(MASTER_CSV),
    '--shortterm-long-csv', str(SHORTTERM_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--oof-years', '2022', '2023', '2024',
    '--test-year', '2025',
    '--device', 'auto', '--threads', '4',
]
print('$', ' '.join(cmd))
log_path = OUTPUT_DIR / 'colab_run.log'
tail = collections.deque(maxlen=80)
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
        tail.append(line)
returncode = process.wait()
if returncode:
    raise RuntimeError(f'실험 실패 (exit code {returncode})\n' + ''.join(tail))
print('완료:', OUTPUT_DIR)

In [ ]:
import json, pandas as pd
from IPython.display import display

metrics = pd.read_csv(OUTPUT_DIR / 'metrics.csv')
display(metrics)
display(metrics[metrics['split'].eq('test_2025')].sort_values('RMSE'))
display(pd.read_csv(OUTPUT_DIR / 'fold_training_summary.csv'))
display(pd.read_csv(OUTPUT_DIR / 'subgroup_metrics_2025.csv'))
with (OUTPUT_DIR / 'calibration_and_weights.json').open(encoding='utf-8') as file:
    recipe = json.load(file)
print(json.dumps(recipe, ensure_ascii=False, indent=2))